# Ingeniería de características — House Prices

Construye el dataset de entrada para el notebook de modelado. Todo el procesamiento vive en
**un solo `Pipeline` de scikit-learn** armado con transformadores de **feature-engine**, que
se ajusta con train y se aplica igual a test.

## Las cinco etapas

| # | Etapa | Qué hace |
|---|---|---|
| 1 | Imputación | Rellena faltantes de numéricas y categóricas |
| 2 | Codificación | Pasa las categóricas a número (ordinal y one-hot) |
| 3 | Outliers | Recorta los valores extremos de las continuas |
| 4 | Transformación | Corrige el sesgo de las continuas que lo necesitan |
| 5 | Escalado | Deja todas las numéricas en la misma escala |

## Decisiones de diseño

- **Se parte de `data/raw`,** no del dataset limpio del notebook 2. Así el pipeline es el
  único objeto que define el preprocesamiento: se aplica igual a train, a test y a cualquier
  dato nuevo. El notebook 2 queda como prototipo exploratorio de esas mismas reglas.
- **La selección de features se recalcula aquí** sobre los datos ya codificados. La del
  notebook 1 se había calculado sobre texto crudo, y siete de sus diez «categóricas» pasan a
  ser ordinales numéricas en cuanto se codifican.
- **Nada se ajusta con test.** Medianas, modas, cuantiles de recorte, parámetros de
  Yeo-Johnson y medias del escalado salen solo de train. Mezclar los dos conjuntos sería fuga
  de datos y las métricas de validación saldrían optimistas.

**Entrada:** `data/raw/{train,test}.csv` · **Salida:** `data/processed/` + el pipeline
ajustado en `models/`

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml

# El paquete del proyecto vive en la raíz, un nivel arriba de notebooks/.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from feature_engine.encoding import OneHotEncoder
from feature_engine.imputation import (
    AddMissingIndicator,
    ArbitraryNumberImputer,
    CategoricalImputer,
    MeanMedianImputer,
    RandomSampleImputer,
)
from feature_engine.outliers import Winsorizer
from feature_engine.preprocessing import MatchVariables
from feature_engine.transformation import LogCpTransformer, YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from house_pricing.features import RellenarAnioGaraje

RAW_DIR = RAIZ / "data" / "raw"
PROCESSED_DIR = RAIZ / "data" / "processed"
MODELS_DIR = RAIZ / "models"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
SEMILLA = 42

print("raíz del proyecto:", RAIZ)

2026-11-04 20:44:01.387 | INFO     | house_pricing.config:<module>:11 - PROJ_ROOT path is: C:\Users\dgarc\OneDrive\Escritorio\Maestria\4T_2026\Product_Mang\Clase_Cloud\house_pricing


raíz del proyecto: c:\Users\dgarc\OneDrive\Escritorio\Maestria\4T_2026\Product_Mang\Clase_Cloud\house_pricing


## 1. Carga y tratamiento de outliers — primera parte: filas

El tratamiento de outliers se hace en **dos momentos distintos**, y conviene no confundirlos.

Aquí se eliminan **filas enteras**, y eso no puede ir dentro del `Pipeline`: un pipeline de
scikit-learn transforma `X` fila por fila y debe devolver tantas filas como recibe. Si borrara
filas de test, nos quedaríamos sin predicciones que entregar.

Se eliminan solo dos, y solo de train. Los Id **524** y **1299** son casas de calidad 10, de
más de 4000 pies², vendidas por menos de 200 mil con `SaleCondition = Partial` (obra sin
terminar al momento de la venta): el precio no refleja el inmueble. Los otros dos casos
grandes, Id 692 y 1183, cuestan unos 750 mil — son coherentes y sostienen el extremo alto del
rango, así que se conservan.

El recorte de **valores** extremos (capping) sí va dentro del pipeline, en la etapa 3.

In [2]:
train_crudo = pd.read_csv(RAW_DIR / "train.csv")
test_crudo = pd.read_csv(RAW_DIR / "test.csv")

IDS_OUTLIERS = [524, 1299]
antes = len(train_crudo)
train_crudo = train_crudo[~train_crudo["Id"].isin(IDS_OUTLIERS)].reset_index(drop=True)

print(f"train: {antes} -> {len(train_crudo)} filas ({antes - len(train_crudo)} eliminadas)")
print(f"test:  {len(test_crudo)} filas (intacto: hay que predecir todas)")

train: 1460 -> 1458 filas (2 eliminadas)
test:  1459 filas (intacto: hay que predecir todas)


## 2. Re-selección de features

La selección del notebook 1 se calculó sobre los datos crudos, donde `ExterQual` o
`KitchenQual` eran texto y se medían con η². Al codificarlas como ordinales pasan a ser
numéricas y deben competir con Pearson, igual que el resto.

Esta celda replica la codificación mínima necesaria —marcar los faltantes estructurales y
aplicar las escalas ordinales— **solo para medir**. El resultado son las listas de variables;
el procesamiento de verdad lo hace el pipeline, partiendo otra vez de los datos crudos.

Se toma el **top 10 de cada grupo**: continuas y discretas por `|correlación de Pearson|`,
categóricas por η² (proporción de la varianza del precio que explica la variable).

In [3]:
# NA estructural = ausencia de la característica, no dato perdido.
ESTRUCTURAL_CAT = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
                   "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish", "GarageQual",
                   "GarageCond", "PoolQC", "Fence", "MiscFeature", "MasVnrType"]
ESTRUCTURAL_NUM = ["MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
                   "BsmtFullBath", "BsmtHalfBath", "GarageCars", "GarageArea"]

CALIDAD = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ESCALAS_ORDINALES = {c: CALIDAD for c in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                                          "HeatingQC", "KitchenQual", "FireplaceQu",
                                          "GarageQual", "GarageCond", "PoolQC"]}
ESCALAS_ORDINALES.update({
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Fence": {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Functional": {"Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "LotShape": {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "LandSlope": {"Sev": 1, "Mod": 2, "Gtl": 3},
    "PavedDrive": {"N": 0, "P": 1, "Y": 2},
    "CentralAir": {"N": 0, "Y": 1},
})

# Copia auxiliar, solo para medir.
aux = train_crudo.copy()
for c in ESTRUCTURAL_CAT:
    aux[c] = aux[c].fillna("None")
for c in ESTRUCTURAL_NUM:
    aux[c] = aux[c].fillna(0)
aux["GarageYrBlt"] = aux["GarageYrBlt"].fillna(aux["YearBuilt"])
for col, mapa in ESCALAS_ORDINALES.items():
    aux[col] = aux[col].map(mapa)

y_aux = aux["SalePrice"]


def eta_cuadrado(serie, objetivo):
    """Proporción de la varianza del objetivo que explica la variable categórica."""
    g = serie.fillna("None")
    grupos = [objetivo[g == nivel].to_numpy() for nivel in g.unique()]
    media = objetivo.mean()
    ss_total = ((objetivo - media) ** 2).sum()
    ss_entre = sum(len(gr) * (gr.mean() - media) ** 2 for gr in grupos)
    return ss_entre / ss_total


numericas = aux.select_dtypes("number").columns.drop(["Id", "SalePrice"])
cardinalidad = aux[numericas].nunique()
grupo_continuas = cardinalidad[cardinalidad > 15].index
grupo_discretas = cardinalidad[cardinalidad <= 15].index
grupo_nominales = aux.select_dtypes(exclude="number").columns


def top10_pearson(cols):
    r = pd.Series({c: aux[c].corr(y_aux) for c in cols})
    return r.reindex(r.abs().sort_values(ascending=False).index).head(10)


CONTINUAS = top10_pearson(grupo_continuas)
DISCRETAS = top10_pearson(grupo_discretas)
CATEGORICAS = pd.Series(
    {c: eta_cuadrado(aux[c], y_aux) for c in grupo_nominales}
).sort_values(ascending=False).head(10)

seleccion = pd.concat([
    CONTINUAS.rename("metrica").to_frame().assign(grupo="continua"),
    DISCRETAS.rename("metrica").to_frame().assign(grupo="discreta"),
    CATEGORICAS.rename("metrica").to_frame().assign(grupo="categorica"),
]).round(3)

CONTINUAS, DISCRETAS, CATEGORICAS = (list(CONTINUAS.index), list(DISCRETAS.index),
                                     list(CATEGORICAS.index))
FEATURES = CONTINUAS + DISCRETAS + CATEGORICAS

assert len(set(FEATURES)) == 30, "las 30 features deben ser únicas"
print(f"{len(FEATURES)} features seleccionadas, todas únicas ✓")
seleccion

30 features seleccionadas, todas únicas ✓


,metrica,grupo
GrLivArea,0.735,continua
TotalBsmtSF,0.651,continua
1stFlrSF,0.632,continua
GarageArea,0.629,continua
YearBuilt,0.524,continua
GarageYrBlt,0.509,continua
YearRemodAdd,0.508,continua
MasVnrArea,0.478,continua
BsmtFinSF1,0.409,continua
LotFrontage,0.371,continua


## 3. Diagnóstico de faltantes y regla de imputación

La regla acordada se aplica según el **porcentaje de faltantes en train**:

| Faltantes | Tratamiento |
|---|---|
| ≤ 5 % | Mediana (numéricas) o moda (categóricas) |
| 5 % – 20 % | Muestreo aleatorio de los valores observados, que conserva la distribución real |
| > 20 % | Se descarta la variable |

**La regla se aplica solo a los faltantes genuinos.** En este dataset, 17 de las 19 columnas
con `NA` codifican *ausencia de la característica*: `PoolQC` nulo significa «sin piscina»,
`FireplaceQu` nulo significa «sin chimenea». Tratarlos como huecos tendría dos efectos
absurdos: el muestreo aleatorio le asignaría calidad de garaje a las casas sin garaje, y la
regla del 20 % descartaría `FireplaceQu`, que es la sexta categórica más fuerte del dataset.

Por eso los estructurales se marcan primero como `"None"` o `0` —la ausencia es información,
no un hueco— y recién después se miden los porcentajes.

El muestreo aleatorio usa semilla fija para que el resultado sea reproducible.

In [4]:
# Faltantes genuinos = los que quedan después de marcar los estructurales.
estructural_sel_cat = [c for c in ESTRUCTURAL_CAT if c in FEATURES]
estructural_sel_num = [c for c in ESTRUCTURAL_NUM if c in FEATURES]

genuinos = train_crudo[FEATURES].copy()
for c in estructural_sel_cat:
    genuinos[c] = genuinos[c].fillna("None")
for c in estructural_sel_num:
    genuinos[c] = genuinos[c].fillna(0)
genuinos["GarageYrBlt"] = genuinos["GarageYrBlt"].fillna(train_crudo["YearBuilt"])

pct_na = (genuinos.isna().mean() * 100).round(2)


def banda(p):
    if p > 20:
        return "descartar"
    if p > 5:
        return "aleatorio"
    return "mediana/moda"


def es_texto(col):
    """pandas 3 usa el dtype `str`, no `object`, así que no sirve comparar con 'object'."""
    return not pd.api.types.is_numeric_dtype(genuinos[col])


diagnostico = pd.DataFrame({
    "pct_na_genuino": pct_na,
    "banda": pct_na.map(banda),
    "tipo": ["categorica" if es_texto(c) else "numerica" for c in pct_na.index],
    "estructural": [c in estructural_sel_cat + estructural_sel_num for c in pct_na.index],
})

A_DESCARTAR = diagnostico.index[diagnostico["banda"] == "descartar"].tolist()
A_ALEATORIO = diagnostico.index[diagnostico["banda"] == "aleatorio"].tolist()

print("variables a descartar (>20 %):", A_DESCARTAR or "ninguna")
print("variables con muestreo aleatorio (5-20 %):", A_ALEATORIO or "ninguna")
diagnostico[diagnostico["pct_na_genuino"] > 0].sort_values("pct_na_genuino", ascending=False)

variables a descartar (>20 %): ninguna
variables con muestreo aleatorio (5-20 %): ['LotFrontage']


,pct_na_genuino,banda,tipo,estructural
LotFrontage,17.76,aleatorio,numerica,False


## 4. Etapa 1 — Imputación

Seis pasos, en este orden:

1. **`AddMissingIndicator`** crea `GarageYrBlt_na` *antes* de rellenar nada. Es la memoria de
   qué casas no tenían garaje; sin ella, esa información se perdería al imputar.
2. **`RellenarAnioGaraje`** pone `YearBuilt` donde falta el año del garaje. Un 0 inventaría
   casas construidas en el año cero y rompería cualquier modelo lineal; `YearBuilt`
   correlaciona 0.83 con esta columna. Vive en `house_pricing/features.py` para que el
   pipeline serializado se pueda recargar desde otro proceso.
3. **Categóricas estructurales** → la categoría `"None"`.
4. **Numéricas estructurales** → `0` (sin sótano son 0 pies² de sótano).
5. **Banda 5-20 %** → `RandomSampleImputer`, que toma valores al azar de los observados y por
   eso conserva la forma de la distribución, cosa que la mediana no hace.
6. **Banda ≤ 5 %** → mediana para numéricas, moda para categóricas.

El paso 6 cubre también las columnas que fallan **solo en test** (`MSZoning`, `SaleType`,
`Exterior1st`…): en train tienen 0 % de faltantes, así que caen en la primera banda, y el
imputador aprende igualmente su mediana o moda de train para aplicarla cuando aparezcan.

In [5]:
numericas_sel = [c for c in CONTINUAS + DISCRETAS if not es_texto(c)]
categoricas_sel = [c for c in FEATURES if es_texto(c)]

# Banda <=5 %: todo lo que no sea estructural ni haya caído en la banda aleatoria.
mediana_hasta_5 = [c for c in numericas_sel if c not in estructural_sel_num + A_ALEATORIO]
moda_hasta_5 = [c for c in categoricas_sel if c not in estructural_sel_cat + A_ALEATORIO]

PASOS_IMPUTACION = [
    ("indicador_sin_garaje", AddMissingIndicator(variables=["GarageYrBlt"])),
    ("anio_garaje", RellenarAnioGaraje()),
    ("estructural_cat", CategoricalImputer(imputation_method="missing", fill_value="None",
                                           variables=estructural_sel_cat)),
    ("estructural_num", ArbitraryNumberImputer(arbitrary_number=0,
                                               variables=estructural_sel_num)),
    ("aleatorio_5_a_20", RandomSampleImputer(variables=A_ALEATORIO, random_state=SEMILLA)),
    ("mediana_hasta_5", MeanMedianImputer(imputation_method="median",
                                          variables=mediana_hasta_5)),
    ("moda_hasta_5", CategoricalImputer(imputation_method="frequent", variables=moda_hasta_5)),
]

for nombre, _ in PASOS_IMPUTACION:
    print(" ", nombre)
print(f"\nestructurales: {len(estructural_sel_cat)} cat + {len(estructural_sel_num)} num")
print(f"muestreo aleatorio: {A_ALEATORIO}")
print(f"mediana: {len(mediana_hasta_5)} · moda: {len(moda_hasta_5)}")

  indicador_sin_garaje
  anio_garaje
  estructural_cat
  estructural_num
  aleatorio_5_a_20
  mediana_hasta_5
  moda_hasta_5

estructurales: 5 cat + 5 num
muestreo aleatorio: ['LotFrontage']
mediana: 9 · moda: 10


## 5. Etapa 2 — Codificación de categóricas

Dos mecanismos distintos, según la variable tenga orden o no.

**Ordinal, para las escalas de calidad.** `ExterQual` vale `Po < Fa < TA < Gd < Ex`: ese orden
es información y hay que conservarlo. Se usa el `OrdinalEncoder` de scikit-learn con las
categorías escritas a mano, envuelto en el `SklearnTransformerWrapper` de feature-engine para
que actúe solo sobre esas columnas y devuelva un DataFrame. No se usa el `OrdinalEncoder` de
feature-engine porque ese ordena por frecuencia o por media del target, y aquí el orden
correcto lo da el diccionario de datos, no los datos.

**One-hot, para las nominales.** `Neighborhood` no tiene orden: numerarlo de 1 a 25 le
inventaría una escala. Se genera una columna binaria por nivel.

`handle_unknown="use_encoded_value"` protege contra niveles que aparezcan en test y no en
train. Se conservan todas las dummies (`drop_last=False`): con Ridge o Lasso la regularización
maneja la colinealidad y así ningún nivel queda como referencia arbitraria. Para una regresión
lineal sin regularización habría que poner `drop_last=True`.

In [6]:
ORDINALES = [c for c in ["ExterQual", "KitchenQual", "BsmtQual", "FireplaceQu",
                         "GarageFinish", "HeatingQC", "ExterCond", "BsmtCond",
                         "BsmtExposure", "BsmtFinType1"] if c in FEATURES]
NOMINALES = [c for c in categoricas_sel if c not in ORDINALES]

# Las escalas, en orden de peor a mejor. El índice resultante es el valor codificado.
ORDEN = {c: list(ESCALAS_ORDINALES[c].keys()) for c in ORDINALES}

PASOS_CODIFICACION = [
    ("ordinal", SklearnTransformerWrapper(
        transformer=OrdinalEncoder(categories=[ORDEN[c] for c in ORDINALES],
                                   handle_unknown="use_encoded_value", unknown_value=-1),
        variables=ORDINALES)),
    ("one_hot", OneHotEncoder(variables=NOMINALES, drop_last=False)),
]

print(f"ordinales ({len(ORDINALES)}):", ORDINALES)
print(f"nominales ({len(NOMINALES)}):", NOMINALES)
print("\nescalas aplicadas:")
for c in ORDINALES:
    print(f"  {c:14} {' < '.join(ORDEN[c])}")
niveles = sum(train_crudo[c].fillna("None").nunique() for c in NOMINALES)
print(f"\ndummies que generará el one-hot: ~{niveles}")

ordinales (5): ['ExterQual', 'KitchenQual', 'BsmtQual', 'FireplaceQu', 'GarageFinish']
nominales (10): ['Neighborhood', 'Foundation', 'GarageType', 'MasVnrType', 'Exterior2nd', 'Exterior1st', 'SaleType', 'SaleCondition', 'MSZoning', 'HouseStyle']

escalas aplicadas:
  ExterQual      None < Po < Fa < TA < Gd < Ex
  KitchenQual    None < Po < Fa < TA < Gd < Ex
  BsmtQual       None < Po < Fa < TA < Gd < Ex
  FireplaceQu    None < Po < Fa < TA < Gd < Ex
  GarageFinish   None < Unf < RFn < Fin

dummies que generará el one-hot: ~101


## 6. Etapa 3 — Outliers, segunda parte: recorte de valores

Mientras que en la etapa inicial se eliminaron filas, aquí se **recortan valores**: todo lo
que quede por debajo del percentil 1 o por encima del 99 se lleva a ese límite. Esto sí puede
ir dentro del pipeline, porque no cambia el número de filas y los límites se aprenden de
train y se aplican tal cual a test.

Recortar en vez de eliminar conserva la fila entera: una casa con un `LotArea` disparatado
sigue aportando su superficie, su barrio y su calidad. Es lo contrario de borrarla.

Solo se recortan las **continuas**, y se excluyen los años: un `YearBuilt` de 1880 es una casa
antigua de verdad, no un error de captura, y aplastarlo contra el percentil 1 falsearía el
dato. Las discretas tampoco entran, porque sus valores extremos son niveles legítimos
(`GarageCars = 4` es una casa con cuatro plazas, no un outlier).

In [7]:
A_RECORTAR = [c for c in CONTINUAS if not c.startswith("Year") and c != "GarageYrBlt"]

PASOS_OUTLIERS = [
    ("recorte_percentiles", Winsorizer(capping_method="quantiles", tail="both", fold=0.01,
                                       variables=A_RECORTAR)),
]

print(f"se recortan {len(A_RECORTAR)} continuas al 1 % de cada cola:")
print(" ", A_RECORTAR)
print("\nse excluyen (los años no son errores):",
      [c for c in CONTINUAS if c not in A_RECORTAR])

limites = train_crudo[A_RECORTAR].quantile([0.01, 0.99]).T
limites.columns = ["percentil_1", "percentil_99"]
limites.assign(min=train_crudo[A_RECORTAR].min(), max=train_crudo[A_RECORTAR].max()).round(1)

se recortan 7 continuas al 1 % de cada cola:
  ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'GarageArea', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']

se excluyen (los años no son errores): ['YearBuilt', 'GarageYrBlt', 'YearRemodAdd']


,percentil_1,percentil_99,min,max
GrLivArea,692.1,3083.7,334.0,4476.0
TotalBsmtSF,0.0,2136.0,0.0,3206.0
1stFlrSF,520.0,2200.7,334.0,3228.0
GarageArea,0.0,988.2,0.0,1390.0
MasVnrArea,0.0,780.2,0.0,1600.0
BsmtFinSF1,0.0,1567.0,0.0,2188.0
LotFrontage,21.0,140.0,21.0,313.0


## 7. Etapa 4 — Transformación de las continuas

No todas las variables sesgadas se tratan igual. El sesgo tiene causas distintas y cada una
pide otra herramienta.

**Logaritmo, para las sesgadas sin ceros.** `GrLivArea` y `1stFlrSF` tienen cola derecha larga
y ningún valor en cero: `log(x+1)` las endereza limpiamente.

**Yeo-Johnson, para `LotFrontage`.** Es la alternativa cuando el logaritmo se pasa de largo:
estima el exponente óptimo desde los datos en vez de imponer uno. Con logaritmo, `LotFrontage`
pasaba de sesgo +1.54 a −0.96, o sea volcada al otro lado; con Yeo-Johnson queda en 0.06.

**Las variables con ceros se dejan como están.** `TotalBsmtSF`, `GarageArea`, `MasVnrArea` y
`BsmtFinSF1` tienen entre 2 % y 59 % de ceros, y aplicarles logaritmo las **empeora** de forma
medible: `TotalBsmtSF` pierde correlación con el precio (0.614 → 0.373) y su sesgo pasa de
1.52 a −5.15. El motivo es que los ceros caen en `log(1) = 0` mientras el resto aterriza cerca
de `log(1000) ≈ 6.9`, abriendo un abismo artificial entre «sin sótano» y «con sótano». Ese
pico de masa en el cero no lo arregla ninguna transformación monótona: habría que separarlo en
un indicador binario más la magnitud, que es creación de variables y queda fuera de las cinco
etapas de este notebook.

Los años tampoco se transforman. Lo que les vendría bien es convertirlos a antigüedad
(`YrSold - YearBuilt`), pero eso también es creación de variables.

In [8]:
LOG = [c for c in ["GrLivArea", "1stFlrSF"] if c in CONTINUAS]
YEO = [c for c in ["LotFrontage"] if c in CONTINUAS]

PASOS_TRANSFORMACION = [
    ("log_sin_ceros", LogCpTransformer(variables=LOG, C=1)),
    ("yeo_johnson", YeoJohnsonTransformer(variables=YEO)),
]

sin_transformar = [c for c in CONTINUAS if c not in LOG + YEO]
diag = pd.DataFrame({
    "sesgo": train_crudo[CONTINUAS].skew().round(2),
    "pct_ceros": (train_crudo[CONTINUAS].eq(0).mean() * 100).round(1),
    "tratamiento": ["log1p" if c in LOG else "yeo-johnson" if c in YEO else "ninguno"
                    for c in CONTINUAS],
})
print("log1p:", LOG, "| yeo-johnson:", YEO)
print("sin transformar:", sin_transformar)
diag.sort_values("sesgo", ascending=False)

log1p: ['GrLivArea', '1stFlrSF'] | yeo-johnson: ['LotFrontage']
sin transformar: ['TotalBsmtSF', 'GarageArea', 'YearBuilt', 'GarageYrBlt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1']


,sesgo,pct_ceros,tratamiento
MasVnrArea,2.69,59.1,ninguno
LotFrontage,1.54,0.0,yeo-johnson
GrLivArea,1.01,0.0,log1p
1stFlrSF,0.89,0.0,log1p
BsmtFinSF1,0.76,32.0,ninguno
TotalBsmtSF,0.51,2.5,ninguno
GarageArea,0.13,5.6,ninguno
YearRemodAdd,-0.50,0.0,ninguno
YearBuilt,-0.61,0.0,ninguno
GarageYrBlt,-0.65,0.0,ninguno


## 8. Etapa 5 — Escalado

`StandardScaler` deja cada variable con media 0 y desviación 1. Importa porque Ridge y Lasso
penalizan el tamaño de los coeficientes: sin escalar, `LotArea` (que vale decenas de miles)
recibiría un castigo desproporcionado frente a `OverallQual` (que va de 1 a 10), y la
regularización terminaría eligiendo variables por sus unidades en vez de por su utilidad.

Se escalan las 20 continuas y discretas. **Las dummies del one-hot y el indicador
`GarageYrBlt_na` se dejan en 0/1**: ya están en una escala común y acotada, y escalarlas haría
que cada columna dependiera de qué tan frecuente es su nivel.

Va al final por necesidad: escalar antes del recorte o del logaritmo no serviría de nada,
porque esas operaciones vuelven a cambiar la escala.

In [9]:
A_ESCALAR = CONTINUAS + DISCRETAS

PASOS_ESCALADO = [
    ("escalado", SklearnTransformerWrapper(transformer=StandardScaler(), variables=A_ESCALAR)),
]

print(f"se escalan {len(A_ESCALAR)} variables (10 continuas + 10 discretas)")
print("no se escalan: las dummies del one-hot y el indicador GarageYrBlt_na")

se escalan 20 variables (10 continuas + 10 discretas)
no se escalan: las dummies del one-hot y el indicador GarageYrBlt_na


## 9. El pipeline completo

Las cinco etapas se encadenan en un único objeto. Esto no es cosmético: un `Pipeline`
garantiza que cada transformador aprenda **solo de train** —en `fit`— y que test reciba
exactamente los mismos parámetros, sin posibilidad de mezclar los dos conjuntos por descuido.

También convierte el preprocesamiento en algo que se guarda y se vuelve a cargar: el mismo
objeto que produjo este dataset es el que transformará datos nuevos el día de mañana.

In [10]:
pipeline = Pipeline(
    PASOS_IMPUTACION
    + PASOS_CODIFICACION
    + PASOS_OUTLIERS
    + PASOS_TRANSFORMACION
    + PASOS_ESCALADO
)

print(f"{len(pipeline.steps)} pasos:\n")
for i, (nombre, transformador) in enumerate(pipeline.steps, 1):
    print(f"{i:2}. {nombre:22} {type(transformador).__name__}")
pipeline

13 pasos:

 1. indicador_sin_garaje   AddMissingIndicator
 2. anio_garaje            RellenarAnioGaraje
 3. estructural_cat        CategoricalImputer
 4. estructural_num        ArbitraryNumberImputer
 5. aleatorio_5_a_20       RandomSampleImputer
 6. mediana_hasta_5        MeanMedianImputer
 7. moda_hasta_5           CategoricalImputer
 8. ordinal                SklearnTransformerWrapper
 9. one_hot                OneHotEncoder
10. recorte_percentiles    Winsorizer
11. log_sin_ceros          LogCpTransformer
12. yeo_johnson            YeoJohnsonTransformer
13. escalado               SklearnTransformerWrapper


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('indicador_sin_garaje', ...), ('anio_garaje', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,variables,['GarageYrBlt']
,missing_only,True
,anio_maximo,2010
,fill_value,'None'
,variables,"['BsmtQual', 'FireplaceQu', ...]"
,imputation_method,'missing'
,return_object,False


## 10. Ajuste y verificación

`fit_transform` sobre train y `transform` sobre test. `MatchVariables` alinea las columnas de
test con las de train: si un nivel de `Neighborhood` no aparece en test, su dummy no se
generaría y el modelo recibiría una matriz con otra forma; este paso la añade en cero.

Las comprobaciones que siguen son `assert`: si alguna falla, el notebook se detiene en vez de
dejar pasar datos malos al modelado.

In [11]:
X_train = train_crudo[FEATURES].copy()
X_test = test_crudo[FEATURES].copy()
y_train = train_crudo["SalePrice"].copy()

X_train_t = pipeline.fit_transform(X_train, y_train)
X_test_t = pipeline.transform(X_test)
X_test_t = MatchVariables(missing_values="ignore").fit(X_train_t).transform(X_test_t)

# 1) Sin faltantes.
assert X_train_t.isna().sum().sum() == 0, "quedan NaN en train"
assert X_test_t.isna().sum().sum() == 0, "quedan NaN en test"
# 2) Mismas columnas y en el mismo orden.
assert list(X_train_t.columns) == list(X_test_t.columns), "columnas descompensadas"
# 3) Todo numérico: ningún modelo acepta texto.
no_num = [c for c in X_train_t.columns if not pd.api.types.is_numeric_dtype(X_train_t[c])]
assert not no_num, f"columnas no numéricas: {no_num}"
# 4) Test conserva sus 1459 filas.
assert len(X_test_t) == 1459, f"test perdió filas: {len(X_test_t)}"
# 5) El escalado dejó las continuas centradas.
medias = X_train_t[A_ESCALAR].mean().abs()
assert (medias < 1e-9).all(), "el escalado no centró las variables"

print("las 5 verificaciones pasan ✓")
print(f"train {X_train_t.shape} · test {X_test_t.shape}")
print(f"{len(FEATURES)} features de entrada -> {X_train_t.shape[1]} columnas de salida")

las 5 verificaciones pasan ✓
train (1458, 122) · test (1459, 122)
30 features de entrada -> 122 columnas de salida


In [12]:
# Efecto de las transformaciones sobre el sesgo.
comparacion = pd.DataFrame({
    "sesgo_antes": train_crudo[LOG + YEO].skew(),
    "sesgo_despues": X_train_t[LOG + YEO].skew(),
    "tratamiento": ["log1p"] * len(LOG) + ["yeo-johnson"] * len(YEO),
}).round(2)
print(comparacion.to_string())

print(f"\ntarget: sesgo {y_train.skew():.2f} -> log1p {np.log1p(y_train).skew():.2f}")
X_train_t.head()

             sesgo_antes  sesgo_despues  tratamiento
GrLivArea           1.01          -0.02        log1p
1stFlrSF            0.89           0.02        log1p
LotFrontage         1.54           0.06  yeo-johnson

target: sesgo 1.88 -> log1p 0.12


,GrLivArea,TotalBsmtSF,1stFlrSF,GarageArea,YearBuilt,GarageYrBlt,YearRemodAdd,MasVnrArea,BsmtFinSF1,LotFrontage,OverallQual,ExterQual,KitchenQual,GarageCars,BsmtQual,FullBath,GarageFinish,TotRmsAbvGrd,FireplaceQu,Fireplaces,GarageYrBlt_na,Neighborhood_CollgCr,Neighborhood_Veenker,Neighborhood_Crawfor,Neighborhood_NoRidge,Neighborhood_Mitchel,Neighborhood_Somerst,Neighborhood_NWAmes,Neighborhood_OldTown,Neighborhood_BrkSide,Neighborhood_Sawyer,Neighborhood_NridgHt,Neighborhood_NAmes,Neighborhood_SawyerW,Neighborhood_IDOTRR,Neighborhood_MeadowV,Neighborhood_Edwards,Neighborhood_Timber,Neighborhood_Gilbert,Neighborhood_StoneBr,Neighborhood_ClearCr,Neighborhood_NPkVill,Neighborhood_Blmngtn,Neighborhood_BrDale,Neighborhood_SWISU,Neighborhood_Blueste,Foundation_PConc,Foundation_CBlock,Foundation_BrkTil,Foundation_Wood,Foundation_Slab,Foundation_Stone,GarageType_Attchd,GarageType_Detchd,GarageType_BuiltIn,GarageType_CarPort,GarageType_None,GarageType_Basment,GarageType_2Types,MasVnrType_BrkFace,...,MasVnrType_BrkCmn,Exterior2nd_VinylSd,Exterior2nd_MetalSd,Exterior2nd_Wd Shng,Exterior2nd_HdBoard,Exterior2nd_Plywood,Exterior2nd_Wd Sdng,Exterior2nd_CmentBd,Exterior2nd_BrkFace,Exterior2nd_Stucco,Exterior2nd_AsbShng,Exterior2nd_Brk Cmn,Exterior2nd_ImStucc,Exterior2nd_AsphShn,Exterior2nd_Stone,Exterior2nd_Other,Exterior2nd_CBlock,Exterior1st_VinylSd,Exterior1st_MetalSd,Exterior1st_Wd Sdng,Exterior1st_HdBoard,Exterior1st_BrkFace,Exterior1st_WdShing,Exterior1st_CemntBd,Exterior1st_Plywood,Exterior1st_AsbShng,Exterior1st_Stucco,Exterior1st_BrkComm,Exterior1st_AsphShn,Exterior1st_Stone,Exterior1st_ImStucc,Exterior1st_CBlock,SaleType_WD,SaleType_New,SaleType_COD,SaleType_ConLD,SaleType_ConLI,SaleType_CWD,SaleType_ConLw,SaleType_Con,SaleType_Oth,SaleCondition_Normal,SaleCondition_Abnorml,SaleCondition_Partial,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,MSZoning_RL,MSZoning_RM,MSZoning_C (all),MSZoning_FV,MSZoning_RH,HouseStyle_2Story,HouseStyle_1Story,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_SFoyer,HouseStyle_SLvl,HouseStyle_2.5Unf,HouseStyle_2.5Fin
0,0.551031,-0.479939,-0.821813,0.371863,1.052959,1.009293,0.880362,0.583129,0.630161,-0.165905,0.658506,1.061109,0.741127,0.313159,0.586329,0.793546,0.320685,0.921812,-1.006993,-0.952231,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
1,-0.392352,0.530123,0.436413,-0.051300,0.158428,-0.017688,-0.428115,-0.602009,1.267168,0.511054,-0.068293,-0.689001,-0.770150,0.313159,0.586329,0.793546,0.320685,-0.316329,0.650737,0.605965,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
2,0.686081,-0.320718,-0.588140,0.660384,0.986698,0.933220,0.831900,0.377544,0.114935,-0.027070,0.658506,1.061109,0.741127,0.313159,0.586329,0.793546,0.320685,-0.316329,0.650737,0.605965,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
3,0.563718,-0.728723,-0.446829,0.823879,-1.862551,0.819111,-0.718888,-0.602009,-0.517388,-0.401629,0.658506,-0.689001,0.741127,1.652119,-0.556543,-1.025620,-0.800558,0.302742,1.203313,0.605965,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0
4,1.330765,0.239046,0.121012,1.756762,0.953567,0.895184,0.734975,1.514309,0.510722,0.684946,1.385305,1.061109,0.741127,1.652119,0.586329,0.793546,0.320685,1.540882,0.650737,0.605965,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,

## 11. Guardado

Se guardan cuatro cosas:

- **`train_features.csv`** — la matriz transformada con `SalePrice` y `SalePrice_log`. Se
  incluyen las dos versiones del target para que el notebook de modelado elija: entrenar sobre
  el logaritmo (que es la métrica oficial de Kaggle, y lleva el sesgo de 1.88 a 0.12) y
  reportar en la escala original.
- **`test_features.csv`** — la matriz de test, con `Id` para poder armar la entrega.
- **`pipeline_features.joblib`** — el pipeline ajustado. Es lo que permite transformar datos
  nuevos exactamente igual, sin recalcular nada.
- **`features_seleccionadas.yaml`** — las listas de variables, para no tener que volver a
  abrir este notebook y leerlas.

In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

salida_train = X_train_t.copy()
salida_train["SalePrice"] = y_train.to_numpy()
salida_train["SalePrice_log"] = np.log1p(y_train.to_numpy())

salida_test = X_test_t.copy()
salida_test.insert(0, "Id", test_crudo["Id"].to_numpy())

ruta_train = PROCESSED_DIR / "train_features.csv"
ruta_test = PROCESSED_DIR / "test_features.csv"
ruta_pipe = MODELS_DIR / "pipeline_features.joblib"
ruta_yaml = PROCESSED_DIR / "features_seleccionadas.yaml"

salida_train.to_csv(ruta_train, index=False)
salida_test.to_csv(ruta_test, index=False)
joblib.dump(pipeline, ruta_pipe)

with open(ruta_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        {
            "TARGET": "SalePrice",
            "TARGET_LOG": "SalePrice_log",
            "MODEL_FEATURES": FEATURES,
            "FEATURE_GROUPS": {"continuas": CONTINUAS, "discretas": DISCRETAS,
                               "categoricas": CATEGORICAS},
            "COLUMNAS_SALIDA": list(X_train_t.columns),
        },
        f, allow_unicode=True, sort_keys=False,
    )

for r in (ruta_train, ruta_test, ruta_pipe, ruta_yaml):
    print(f"{r.name:28} {r.stat().st_size / 1024:8.1f} KB")

train_features.csv              885.4 KB
test_features.csv               856.6 KB
pipeline_features.joblib         23.8 KB
features_seleccionadas.yaml       3.5 KB


## 12. Resumen y siguiente paso

| Etapa | Qué se aplicó |
|---|---|
| Selección | Top 10 por grupo, recalculado sobre los datos codificados |
| Outliers (filas) | 2 filas fuera de train (Id 524 y 1299); test intacto |
| 1 · Imputación | Estructurales → `"None"`/`0`; banda 5-20 % → muestreo aleatorio; banda ≤5 % → mediana/moda |
| 2 · Codificación | Ordinal con escalas del diccionario de datos + one-hot de las nominales |
| 3 · Outliers (valores) | Recorte al 1 % de cada cola en las continuas, sin tocar los años |
| 4 · Transformación | `log1p` en las sesgadas sin ceros, Yeo-Johnson en `LotFrontage` |
| 5 · Escalado | `StandardScaler` en las 20 numéricas; dummies intactas |

Todo quedó dentro de un `Pipeline` ajustado solo con train y guardado en
`models/pipeline_features.joblib`.

### Lo que quedó deliberadamente fuera

Las cinco etapas pedidas no incluyen **creación** de variables, y hay tres candidatas claras
que convendría evaluar en su momento:

- **Antigüedad** (`YrSold - YearBuilt`), en vez de los años en bruto. La brecha entre Spearman
  y Pearson de `YearBuilt` (0.653 contra 0.523) dice que la relación con el precio es monótona
  pero curva, señal de que la representación actual no es la mejor.
- **Indicador + magnitud** para las variables con muchos ceros (`MasVnrArea`, `BsmtFinSF1`),
  que es la única forma de arreglar ese pico en el cero.
- **Superficie total** y **baños totales**, que suelen superar a sus componentes por separado.

Tampoco se resolvieron los pares redundantes (`GarageArea` ↔ `GarageCars` correlacionan 0.882,
`GarageYrBlt` ↔ `YearBuilt` 0.826). Con Ridge o Lasso no rompen nada, pero inflan la varianza
de los coeficientes y vuelven difícil leer qué variable importa de verdad.

### Siguiente paso

**Notebook `4.0-dg-modelado`**, partiendo de `data/processed/train_features.csv`: validación
cruzada con Ridge, Lasso y un modelo de árboles, entrenando sobre `SalePrice_log` y reportando
RMSE en esa escala, que es la métrica oficial de la competencia.